In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from collections import Counter

import librosa
import librosa.display

import torch
import torchaudio
import torchaudio.transforms as AT
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import timm
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f"PyTorch {torch.__version__} | torchaudio {torchaudio.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

PyTorch 2.10.0+cu128 | torchaudio 2.10.0+cu128
GPU: Tesla T4


In [3]:
SEED = 42
ROOT = "/kaggle/input/competitions/birdclef-2026"
WORK = "/kaggle/working/"
SR = 32000
DURATION = 5
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
F_MIN = 20
F_MAX = 16000
NUM_EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)

In [4]:
# ── 1. SETUP ─────────────────────────────────────────────────
import os
import re
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from collections import Counter
from IPython.display import Audio, display
 
BASE   = Path("/kaggle/input/competitions/birdclef-2026")
TRAIN_AUDIO      = BASE / "train_audio"
TRAIN_SOUNDSCAPES= BASE / "train_soundscapes"
TEST_SOUNDSCAPES = BASE / "test_soundscapes"
 
taxonomy_df  = pd.read_csv(BASE / "taxonomy.csv")
train_df     = pd.read_csv(BASE / "train.csv")
labels_df    = pd.read_csv(BASE / "train_soundscapes_labels.csv")
sample_sub   = pd.read_csv(BASE / "sample_submission.csv")
 
sns.set_theme(style="whitegrid", palette="muted")
BLUE = "#4C72B0"
 
print("taxonomy :", taxonomy_df.shape)
print("train    :", train_df.shape)
print("labels   :", labels_df.shape)
print("sub cols :", list(sample_sub.columns[:5]), "...")

taxonomy : (234, 5)
train    : (35549, 15)
labels   : (1478, 4)
sub cols : ['row_id', '1161364', '116570', '1176823', '1491113'] ...


In [5]:
# ── 6. AUDIO EDA ──────────────────────────────────────────────
# 6a. File inventory
train_audio_files = sorted(TRAIN_AUDIO.rglob("*.ogg"))
soundscape_files  = sorted(TRAIN_SOUNDSCAPES.glob("*.ogg"))
test_files        = sorted(TEST_SOUNDSCAPES.glob("*.ogg"))
 
print(f"\n── AUDIO FILES ──")
print(f"  train_audio files   : {len(train_audio_files)}")
print(f"  train_soundscapes   : {len(soundscape_files)}")


── AUDIO FILES ──
  train_audio files   : 35549
  train_soundscapes   : 10658


In [6]:
# 6c. Soundscape duration spot-check
sc_durations = []
for p in soundscape_files[:10]:
    try:
        y, sr = librosa.load(p, sr=None, mono=True)
        sc_durations.append(librosa.get_duration(y=y, sr=sr))
    except Exception:
        pass
print(f"\nSoundscape durations (first 10): {[round(d,1) for d in sc_durations]}")


Soundscape durations (first 10): [60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0]


In [7]:
import os
from pathlib import Path

esc_path = Path("/kaggle/input/datasets/mmoreaux/environmental-sound-classification-50")
print("Top level:", os.listdir(esc_path))

# Check audio folder
audio_path = esc_path / "audio"
if audio_path.exists():
    files = list(audio_path.rglob("*.wav")) + list(audio_path.rglob("*.ogg"))
    print(f"\nAudio files found: {len(files)}")
    print("Sample files:", [f.name for f in files[:5]])
else:
    # Audio might be nested further
    for root, dirs, files in os.walk(esc_path):
        if any(f.endswith(('.wav', '.ogg', '.mp3')) for f in files):
            print(f"Audio found at: {root}")
            print(f"Sample: {files[:3]}")
            break

Top level: ['audio', 'utils2.py', 'esc50.csv', 'bc_utils.py', 'utils.py']

Audio files found: 6000
Sample files: ['5-257349-A-15.wav', '5-195557-A-19.wav', '2-122820-B-36.wav', '1-115920-A-22.wav', '1-172649-C-40.wav']


In [9]:
# ── Augmentation functions v2 ─────────────────────────────────

def load_noise_sample(noise_files, target_length):
    """Load a random ESC-50 file and fit to target length."""
    if not noise_files:
        return None
    path = np.random.choice(noise_files)
    try:
        y_noise, _ = librosa.load(str(path), sr=CFG.sr, mono=True)
        # Tile if too short
        if len(y_noise) < target_length:
            repeats = int(np.ceil(target_length / len(y_noise)))
            y_noise = np.tile(y_noise, repeats)
        # Random crop
        start = np.random.randint(0, len(y_noise) - target_length + 1)
        return y_noise[start: start + target_length]
    except Exception:
        return None


def augment_waveform_v2(y, training=True, noise_files=None):
    """Enhanced waveform augmentation."""
    if not training:
        return y

    # 1. Random time shift
    if np.random.random() < 0.5:
        shift = np.random.randint(0, CFG.sr)
        y = np.roll(y, shift)

    # 2. Random gain
    if np.random.random() < 0.5:
        gain = 10 ** (np.random.uniform(-6, 6) / 20)
        y = (y * gain).clip(-1.0, 1.0)

    # 3. Gaussian noise
    if np.random.random() < 0.3:
        y = y + np.random.uniform(0.0005, 0.002) * np.random.randn(len(y))

    # 4. Pink noise (1/f noise — more realistic than white)
    if np.random.random() < 0.3:
        white = np.random.randn(len(y))
        freqs = np.fft.rfftfreq(len(y))
        freqs[0] = 1e-6  # avoid div by zero
        pink_filter = 1.0 / np.sqrt(freqs)
        pink = np.fft.irfft(np.fft.rfft(white) * pink_filter, n=len(y))
        pink = pink / (np.abs(pink).max() + 1e-8)
        snr  = np.random.uniform(10, 30)  # dB
        noise_scale = np.sqrt(10 ** (-snr / 10))
        y = y + noise_scale * pink.astype(np.float32)

    # 5. ESC-50 background noise (most important for domain gap)
    if np.random.random() < 0.5 and noise_files:
        bg = load_noise_sample(noise_files, len(y))
        if bg is not None:
            snr = np.random.uniform(5, 20)  # dB — fairly loud background
            signal_rms = np.sqrt(np.mean(y ** 2)) + 1e-8
            noise_rms  = np.sqrt(np.mean(bg ** 2)) + 1e-8
            scale = signal_rms / noise_rms / (10 ** (snr / 20))
            y = y + scale * bg
            y = y.clip(-1.0, 1.0)

    return y.astype(np.float32)

In [10]:
# ── Improved iNat Dataset ─────────────────────────────────────
class InatDatasetV2(Dataset):
    """
    v2 changes:
    - 15s windows
    - Secondary labels with 0.5 weight
    - ESC-50 background noise augmentation
    """
    def __init__(self, df, sub_cols, noise_files=None, training=True):
        self.df          = df.reset_index(drop=True)
        self.sub_cols    = sub_cols
        self.noise_files = noise_files or []
        self.training    = training
        self.label_to_idx = {col: i for i, col in enumerate(sub_cols)}
        self.n_classes   = len(sub_cols)
        print(f"InatDatasetV2: {len(self.df)} recordings | "
              f"noise_files={len(self.noise_files)} | training={training}")

    def __len__(self):
        return len(self.df)

    def _build_label_vector(self, primary, secondary=None):
        vec = np.zeros(self.n_classes, dtype=np.float32)

        # Primary label — full weight 1.0
        p = str(primary).strip()
        if p in self.label_to_idx:
            vec[self.label_to_idx[p]] = 1.0

        # Secondary labels — half weight 0.5
        if secondary is not None:
            try:
                sec_list = eval(str(secondary)) \
                    if isinstance(secondary, str) else []
                for s in sec_list:
                    s = str(s).strip()
                    if s in self.label_to_idx:
                        vec[self.label_to_idx[s]] = max(
                            vec[self.label_to_idx[s]], 0.5
                        )
            except Exception:
                pass
        return vec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = CFG.train_audio_dir / str(row["filename"])

        try:
            total_dur = librosa.get_duration(path=str(audio_path))
        except Exception:
            total_dur = CFG.duration

        # Random window during training, centre during val
        if total_dur <= CFG.duration:
            offset = 0.0
        elif self.training:
            offset = np.random.uniform(0, total_dur - CFG.duration)
        else:
            offset = (total_dur - CFG.duration) / 2

        try:
            y, sr = librosa.load(
                str(audio_path), sr=CFG.sr, mono=True,
                offset=offset, duration=CFG.duration
            )
        except Exception:
            y = np.zeros(CFG.samples, dtype=np.float32)

        # Augment
        y = augment_waveform_v2(
            y, training=self.training, noise_files=self.noise_files
        )

        # PCEN
        spec = audio_to_pcen(y)
        spec = augment_spectrogram(spec, training=self.training)
        spec = torch.tensor(spec).unsqueeze(0)

        # Labels
        label_vec = self._build_label_vector(
            row["primary_label"],
            row.get("secondary_labels", None)
        )
        return spec, torch.tensor(label_vec)

In [11]:
import os, librosa, numpy as np, pandas as pd
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
import time

class CFG:
    sr         = 32000
    duration   = 15
    n_mels     = 128
    fmax       = 16000
    hop_length = 512
    n_fft      = 1024
    samples    = sr * duration
    train_audio_dir = Path("/kaggle/input/competitions/birdclef-2026/train_audio")
    soundscape_dir  = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")

In [12]:
class BirdCLEFModel(nn.Module):
    def __init__(self, n_classes=234, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=pretrained,
            in_chans=1, num_classes=0, global_pool="avg",
        )
        n_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, n_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

class SmoothedBCELoss(nn.Module):
    def __init__(self, smoothing=0.3):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets, weights=None):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        if weights is not None:
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits, targets_smooth, reduction="none"
            ).mean(dim=1)
            return (loss * weights).mean()
        return nn.functional.binary_cross_entropy_with_logits(
            logits, targets_smooth
        )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [13]:
def audio_to_pcen(y, sr=CFG.sr):
    if len(y) < CFG.samples:
        y = np.pad(y, (0, CFG.samples - len(y)), mode="reflect")
    else:
        y = y[:CFG.samples]
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=CFG.n_mels, fmax=CFG.fmax,
        hop_length=CFG.hop_length, n_fft=CFG.n_fft,
    )
    return librosa.pcen(S * (2**31), sr=sr,
                        hop_length=CFG.hop_length).astype(np.float32)

def augment_spectrogram(spec, training=True):
    if not training:
        return spec
    spec = spec.copy()
    n_mels, time_steps = spec.shape
    if np.random.random() < 0.5:
        f_width = np.random.randint(0, 20)
        f_start = np.random.randint(0, max(1, n_mels - f_width))
        spec[f_start:f_start + f_width, :] = 0.0
    if np.random.random() < 0.5:
        t_width = np.random.randint(0, 20)
        t_start = np.random.randint(0, max(1, time_steps - t_width))
        spec[:, t_start:t_start + t_width] = 0.0
    return spec

def load_noise_sample(noise_files, target_length):
    if not noise_files:
        return None
    path = np.random.choice(noise_files)
    try:
        y_noise, _ = librosa.load(str(path), sr=CFG.sr, mono=True)
        if len(y_noise) < target_length:
            y_noise = np.tile(y_noise, int(np.ceil(target_length / len(y_noise))))
        start = np.random.randint(0, len(y_noise) - target_length + 1)
        return y_noise[start: start + target_length]
    except Exception:
        return None

def augment_waveform_v2(y, training=True, noise_files=None):
    if not training:
        return y
    if np.random.random() < 0.5:
        y = np.roll(y, np.random.randint(0, CFG.sr))
    if np.random.random() < 0.5:
        y = (y * 10 ** (np.random.uniform(-6, 6) / 20)).clip(-1.0, 1.0)
    if np.random.random() < 0.3:
        y = y + np.random.uniform(0.0005, 0.002) * np.random.randn(len(y))
    if np.random.random() < 0.3:
        white = np.random.randn(len(y))
        freqs = np.fft.rfftfreq(len(y))
        freqs[0] = 1e-6
        pink = np.fft.irfft(np.fft.rfft(white) / np.sqrt(freqs), n=len(y))
        pink = pink / (np.abs(pink).max() + 1e-8)
        y = y + np.random.uniform(0.001, 0.005) * pink.astype(np.float32)
    if np.random.random() < 0.5 and noise_files:
        bg = load_noise_sample(noise_files, len(y))
        if bg is not None:
            snr = np.random.uniform(5, 20)
            sig_rms = np.sqrt(np.mean(y ** 2)) + 1e-8
            noi_rms = np.sqrt(np.mean(bg ** 2)) + 1e-8
            scale = sig_rms / noi_rms / (10 ** (snr / 20))
            y = (y + scale * bg).clip(-1.0, 1.0)
    return y.astype(np.float32)

print("✓ Functions defined")

✓ Functions defined


In [14]:
esc_candidates = [d for d in os.listdir("/kaggle/input/")
                  if any(x in d.lower() for x in ["esc", "environ", "noise"])]
print("ESC-50 candidates:", esc_candidates)

if esc_candidates:
    ESC50_PATH = f"/kaggle/input/{esc_candidates[0]}"
    noise_files = list(Path(ESC50_PATH).rglob("*.wav")) + \
                  list(Path(ESC50_PATH).rglob("*.ogg"))
    print(f"Noise files found: {len(noise_files)}")
else:
    noise_files = []
    print("No ESC-50 — training without background noise")

ESC-50 candidates: []
No ESC-50 — training without background noise


In [15]:
class InatDatasetV2(Dataset):
    def __init__(self, df, sub_cols, noise_files=None, training=True):
        self.df           = df.reset_index(drop=True)
        self.sub_cols     = sub_cols
        self.noise_files  = noise_files or []
        self.training     = training
        self.label_to_idx = {col: i for i, col in enumerate(sub_cols)}
        self.n_classes    = len(sub_cols)
        print(f"InatDatasetV2: {len(self.df)} recordings | "
              f"noise={len(self.noise_files)} | training={training}")

    def __len__(self):
        return len(self.df)

    def _build_label_vector(self, primary, secondary=None):
        vec = np.zeros(self.n_classes, dtype=np.float32)
        p = str(primary).strip()
        if p in self.label_to_idx:
            vec[self.label_to_idx[p]] = 1.0
        if secondary is not None:
            try:
                for s in (eval(str(secondary)) if isinstance(secondary, str) else []):
                    s = str(s).strip()
                    if s in self.label_to_idx:
                        vec[self.label_to_idx[s]] = max(vec[self.label_to_idx[s]], 0.5)
            except Exception:
                pass
        return vec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = CFG.train_audio_dir / str(row["filename"])
        try:
            total_dur = librosa.get_duration(path=str(audio_path))
        except Exception:
            total_dur = CFG.duration
        if total_dur <= CFG.duration:
            offset = 0.0
        elif self.training:
            offset = np.random.uniform(0, total_dur - CFG.duration)
        else:
            offset = (total_dur - CFG.duration) / 2
        try:
            y, sr = librosa.load(str(audio_path), sr=CFG.sr, mono=True,
                                 offset=offset, duration=CFG.duration)
        except Exception:
            y = np.zeros(CFG.samples, dtype=np.float32)
        y    = augment_waveform_v2(y, self.training, self.noise_files)
        spec = audio_to_pcen(y)
        spec = augment_spectrogram(spec, self.training)
        spec = torch.tensor(spec).unsqueeze(0)
        label_vec = self._build_label_vector(
            row["primary_label"], row.get("secondary_labels")
        )
        return spec, torch.tensor(label_vec)

print("✓ InatDatasetV2 defined")

✓ InatDatasetV2 defined


In [16]:
from sklearn.metrics import roc_auc_score

def compute_auc(all_labels, all_probs):
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    aucs = []
    for i in range(all_labels.shape[1]):
        if all_labels[:, i].sum() > 0:
            try:
                aucs.append(roc_auc_score(all_labels[:, i], all_probs[:, i]))
            except Exception:
                pass
    return np.mean(aucs) if aucs else 0.0

print("✓ compute_auc defined")

✓ compute_auc defined


In [17]:
# ── Reload all data ───────────────────────────────────────────
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/competitions/birdclef-2026")

taxonomy_df  = pd.read_csv(BASE / "taxonomy.csv")
train_df     = pd.read_csv(BASE / "train.csv")
labels_df    = pd.read_csv(BASE / "train_soundscapes_labels.csv")
sample_sub   = pd.read_csv(BASE / "sample_submission.csv")

# Submission target columns
sub_cols = set(sample_sub.columns[1:].tolist())

# iNat filtered dataset
inat_df = train_df[
    (train_df["rating"] >= 3) &
    (train_df["primary_label"].astype(str).isin(sub_cols))
].copy().reset_index(drop=True)

print(f"taxonomy : {taxonomy_df.shape}")
print(f"train_df : {train_df.shape}")
print(f"inat_df  : {inat_df.shape}")
print(f"sub_cols : {len(sub_cols)}")
print(f"sample   : {sample_sub.shape}")

taxonomy : (234, 5)
train_df : (35549, 15)
inat_df  : (21295, 15)
sub_cols : 234
sample   : (3, 235)


In [ ]:
# ── Retrain with v2 dataset ───────────────────────────────────
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

BATCH_SIZE  = 16   # ← reduced from 32, 15s specs are 3× larger
NUM_EPOCHS  = 5
LR          = 1e-3
NUM_WORKERS = 2
VAL_SPLIT   = 0.1

# Fresh model
model = BirdCLEFModel(n_classes=234, pretrained=False).to(device)
criterion = SmoothedBCELoss(smoothing=0.3)

full_ds = InatDatasetV2(
    inat_df, list(sub_cols),
    noise_files=noise_files,
    training=True
)

n_val   = int(len(full_ds) * VAL_SPLIT)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
val_ds.dataset.training = False

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_auc = 0.0
best_epoch = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for batch_idx, (specs, labels) in enumerate(train_loader):
        specs  = specs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(specs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch} | batch {batch_idx+1}/{len(train_loader)} "
                  f"| loss {loss.item():.4f}")

    train_loss /= len(train_loader)

    model.eval()
    val_loss, all_labels, all_probs = 0.0, [], []

    with torch.no_grad():
        for specs, labels in val_loader:
            specs  = specs.to(device)
            labels = labels.to(device)
            logits = model(specs)
            val_loss += criterion(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_auc   = compute_auc(np.vstack(all_labels), np.vstack(all_probs))
    scheduler.step()

    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"val_auc={val_auc:.4f} | lr={scheduler.get_last_lr()[0]:.2e} | "
          f"{time.time()-t0:.0f}s")

    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_auc": val_auc})

    if val_auc > best_auc:
        best_auc   = val_auc
        best_epoch = epoch
        torch.save(model.state_dict(), "/kaggle/working/best_model_v2.pth")
        print(f"  ✓ Saved best model v2 (auc={best_auc:.4f})")

    # Checkpoint every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "best_auc": best_auc,
        "history": history,
    }, "/kaggle/working/checkpoint_latest_v2.pth")

print(f"\nTraining complete. Best AUC={best_auc:.4f} at epoch {best_epoch}")

In [ ]:
import os

# Search all inputs for the file
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "soundscape" in f.lower() and f.endswith(".csv"):
            print(os.path.join(root, f))

In [ ]:
# ── Soundscape fine-tuning v2 ─────────────────────────────────
# Load the v2 weights as starting point

model.load_state_dict(
    torch.load("/kaggle/working/best_model_v2.pth", map_location=device)
)
print(f"✓ Loaded best_model_v2 (iNat AUC=0.7016)")

label_matrix = pd.read_csv(
    "/kaggle/input/datasets/minervasdatalab/soundscape-matrix/soundscape_label_matrix.csv"
)

# Site-aware split
label_matrix["site"] = label_matrix["filename"].apply(
    lambda x: next(
        (p for p in x.split("_") if p.startswith("S") and p[1:].isdigit()), "S00"
    )
)
sites      = label_matrix["site"].unique()
np.random.seed(42)
val_sites  = np.random.choice(sites, size=max(1, len(sites)//5), replace=False)
train_sites = [s for s in sites if s not in val_sites]
train_sc = label_matrix[label_matrix["site"].isin(train_sites)].reset_index(drop=True)
val_sc   = label_matrix[label_matrix["site"].isin(val_sites)].reset_index(drop=True)
print(f"Train sites: {len(train_sites)} | Val sites: {len(val_sites)}")
print(f"Train windows: {len(train_sc)} | Val windows: {len(val_sc)}")

species_cols = [c for c in sub_cols if c in label_matrix.columns]
soundscape_dir = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")

class SoundscapeDataset(torch.utils.data.Dataset):
    def __init__(self, df, species_cols, training=True):
        self.df           = df.reset_index(drop=True)
        self.species_cols = list(species_cols)
        self.training     = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y, sr = librosa.load(
                str(soundscape_dir / row["filename"]),
                sr=CFG.sr, mono=True,
                offset=float(row["t_start"]), duration=CFG.duration
            )
        except Exception:
            y = np.zeros(CFG.samples, dtype=np.float32)

        if self.training and np.random.random() < 0.5:
            y = np.roll(y, np.random.randint(0, CFG.sr // 2))

        spec   = audio_to_pcen(y)
        spec   = augment_spectrogram(spec, self.training)
        spec   = torch.tensor(spec).unsqueeze(0)
        labels = torch.tensor(
            row[self.species_cols].values.astype(np.float32)
        )
        weight = torch.tensor(float(row.get("weight", 1.0)))
        return spec, labels, weight

# Add simple energy-based weights
train_sc["weight"] = 1.0  # uniform for now — energy precompute takes time

train_sc_ds = SoundscapeDataset(train_sc, species_cols, training=True)
val_sc_ds   = SoundscapeDataset(val_sc,   species_cols, training=False)

train_sc_loader = DataLoader(train_sc_ds, batch_size=16, shuffle=True,
                             num_workers=2, pin_memory=True)
val_sc_loader   = DataLoader(val_sc_ds,   batch_size=16, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_sc_loader)} | Val batches: {len(val_sc_loader)}")

FT_EPOCHS  = 5
ft_optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-7)
ft_criterion = SmoothedBCELoss(smoothing=0.2)

best_ft_auc, best_ft_epoch = 0.0, 0

for epoch in range(1, FT_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for specs, labels, weights in train_sc_loader:
        specs, labels, weights = (specs.to(device), labels.to(device),
                                  weights.to(device))
        ft_optimizer.zero_grad()
        loss = ft_criterion(model(specs), labels, weights=weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        ft_optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_sc_loader)
    model.eval()
    val_loss, all_labels, all_probs = 0.0, [], []

    with torch.no_grad():
        for specs, labels, weights in val_sc_loader:
            specs, labels = specs.to(device), labels.to(device)
            logits = model(specs)
            val_loss += ft_criterion(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_sc_loader)
    val_auc   = compute_auc(np.vstack(all_labels), np.vstack(all_probs))
    ft_scheduler.step()

    print(f"FT Epoch {epoch:02d}/{FT_EPOCHS} | train={train_loss:.4f} | "
          f"val={val_loss:.4f} | auc={val_auc:.4f} | "
          f"lr={ft_scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")

    if val_auc > best_ft_auc:
        best_ft_auc, best_ft_epoch = val_auc, epoch
        torch.save(model.state_dict(),
                   "/kaggle/working/best_model_v2_finetuned.pth")
        print(f"  ✓ Saved (auc={best_ft_auc:.4f})")

print(f"\nFine-tuning complete. Best AUC={best_ft_auc:.4f} at epoch {best_ft_epoch}")

In [ ]:
# Verify v2 weights exist
import os
print(os.path.exists("/kaggle/working/best_model_v2.pth"))
print(os.path.getsize("/kaggle/working/best_model_v2.pth")/1024/1024, "MB")

In [ ]:
import os

# Check all available weight files across datasets
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pth"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)/1024/1024
            print(f"{size:.1f} MB  {path}")

In [ ]:
# Load v2 and check predictions aren't degenerate
model.eval()
dummy = torch.randn(4, 1, 128, 313).to(device)
with torch.no_grad():
    out = torch.sigmoid(model(dummy))
print(f"Output mean: {out.mean():.4f}")  
print(f"Output std:  {out.std():.4f}")
# Healthy model: mean ~0.3-0.5, std ~0.1-0.2
# Broken model:  mean ~0.5, std ~0.001 (all same value)

Models

In [ ]:
import os

# Check Models
print("=== BirdNET ONNX (Models) ===")
for root, dirs, files in os.walk("/kaggle/input/models/shadiakiki1/birdnet-analyzer/tflite/birdnet_global_6k_v2.4_model_fp32-1/2"):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}  ({os.path.getsize(path)/1024/1024:.1f} MB)")

print("\n=== BirdNET Analyzer 2023 (Datasets) ===")
for root, dirs, files in os.walk("/kaggle/input/datasets/seshurajup/birdnet-analyzer-2023"):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}  ({os.path.getsize(path)/1024/1024:.1f} MB)")

In [ ]:
# ── TFLite via TensorFlow (already installed on Kaggle) ───────
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

BIRDNET_MODEL = "/kaggle/input/models/shadiakiki1/birdnet-analyzer/tflite/birdnet_global_6k_v2.4_model_fp32-1/2/BirdNET_GLOBAL_6K_V2.4_Model_FP32.tflite"

# Load via tf.lite
interpreter = tf.lite.Interpreter(model_path=BIRDNET_MODEL)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\nInput shape: {input_details[0]['shape']}")
print(f"Input dtype: {input_details[0]['dtype']}")
print(f"\nAll outputs:")
for i, od in enumerate(output_details):
    print(f"  Output {i}: shape={od['shape']} | name={od['name']}")

In [ ]:
# Use correct signature name
infer = birdnet_model.signatures["basic"]

print("Inputs:")
for k, v in infer.structured_input_signature[1].items():
    print(f"  {k}: {v}")

print("\nOutputs:")
for k, v in infer.structured_outputs.items():
    print(f"  {k}: shape={v.shape}")

In [ ]:
# ── Extract intermediate layer as embedding ───────────────────
# Get the keras model from the SavedModel
keras_model = birdnet_model

# Check if it has layers attribute
try:
    print("Layers:")
    for layer in keras_model.layers:
        print(f"  {layer.name}: {layer.output_shape}")
except Exception as e:
    print(f"Not a keras model: {e}")

# Try accessing the underlying keras model differently
try:
    print("\nTrackable objects:")
    for name in dir(keras_model):
        if not name.startswith("_"):
            print(f"  {name}")
except Exception:
    pass

In [ ]:
# ── Access the underlying model ───────────────────────────────
inner_model = birdnet_model.model
print(type(inner_model))

try:
    print("\nLayers:")
    for layer in inner_model.layers:
        print(f"  {layer.name}: output_shape={layer.output_shape}")
except Exception as e:
    print(f"Error: {e}")

# Try calling it directly to get intermediate outputs
try:
    test_input = tf.random.normal([1, 144000])
    output = inner_model(test_input)
    print(f"\nDirect call output shape: {output.shape}")
except Exception as e:
    print(f"Direct call error: {e}")

In [ ]:
# ── Find embedding tensor in v2.4 tflite ─────────────────────
all_tensors = interpreter.get_tensor_details()
print(f"Total tensors: {len(all_tensors)}")
print("\nLast 20 tensors:")
for t in all_tensors[-20:]:
    print(f"  idx={t['index']} | shape={t['shape']} | name={t['name'][:60]}")

In [18]:
!pip install librosa


In [20]:
import librosa

In [ ]:
# ── Use BirdNET scores directly as features ───────────────────
def get_birdnet_scores(y, sr=32000):
    """Get 3337-dim species score vector from BirdNET v2.2."""
    y_48k = librosa.resample(y, orig_sr=sr, target_sr=48000)
    target_len = 144000
    if len(y_48k) < target_len:
        y_48k = np.pad(y_48k, (0, target_len - len(y_48k)))
    else:
        start = (len(y_48k) - target_len) // 2
        y_48k = y_48k[start: start + target_len]
    
    input_tensor = tf.constant(y_48k.reshape(1, -1), dtype=tf.float32)
    output = infer(input_tensor)
    return output["scores"].numpy()[0]  # (3337,)

# Test
test_audio = np.random.randn(160000).astype(np.float32)
scores = get_birdnet_scores(test_audio)
print(f"BirdNET scores shape: {scores.shape}")  # (3337,)
print(f"Score range: {scores.min():.3f} to {scores.max():.3f}")

In [ ]:
import pandas as pd

In [ ]:
# ── Map BirdNET species → our 234 target species ──────────────
# Load BirdNET labels to find which of its 3337 species
# overlap with our 234 competition targets

# Read BirdNET v2.2 labels
with open("/kaggle/input/datasets/seshurajup/birdnet-analyzer-2023/BirdNET-Analyzer/checkpoints/V2.2/BirdNET_GLOBAL_3K_V2.2_Labels.txt") as f:
    birdnet_labels = [line.strip() for line in f.readlines()]

print(f"BirdNET v2.2 labels: {len(birdnet_labels)}")
print(f"Sample labels: {birdnet_labels[:5]}")

# Load our taxonomy to get scientific names
taxonomy_df = pd.read_csv("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
sample_sub  = pd.read_csv("/kaggle/input/competitions/birdclef-2026/sample_submission.csv")
sub_cols    = sample_sub.columns[1:].tolist()

# BirdNET labels format: "Turdus_rufiventris_Rufous-bellied Thrush"
# Extract scientific name part (first two words)
birdnet_sci = {}
for i, label in enumerate(birdnet_labels):
    parts = label.split("_")
    if len(parts) >= 2:
        sci = f"{parts[0]} {parts[1]}"
        birdnet_sci[sci.lower()] = i

# Match against our taxonomy
taxonomy_df["sci_lower"] = taxonomy_df["scientific_name"].str.lower()
taxonomy_df["birdnet_idx"] = taxonomy_df["sci_lower"].map(birdnet_sci)

matched = taxonomy_df[taxonomy_df["birdnet_idx"].notna()]
print(f"\nOur species matched in BirdNET: {len(matched)} / {len(taxonomy_df)}")
print(matched[["scientific_name", "primary_label", "birdnet_idx"]].head(10))

In [ ]:
# ── Fix: match on scientific name with underscore → space ─────
birdnet_sci = {}
for i, label in enumerate(birdnet_labels):
    # Format: "Turdus_rufiventris_Rufous-bellied Thrush"
    parts = label.split("_")
    if len(parts) >= 2:
        sci = f"{parts[0]} {parts[1]}"  # "Turdus rufiventris"
        birdnet_sci[sci.lower()] = i

# Also try matching on common name
birdnet_common = {}
for i, label in enumerate(birdnet_labels):
    parts = label.split("_")
    if len(parts) >= 3:
        common = " ".join(parts[2:])  # everything after genus_species
        birdnet_common[common.lower()] = i

# Match scientific names
taxonomy_df["sci_lower"]    = taxonomy_df["scientific_name"].str.strip().str.lower()
taxonomy_df["common_lower"] = taxonomy_df["common_name"].str.strip().str.lower()

taxonomy_df["birdnet_idx"] = taxonomy_df["sci_lower"].map(birdnet_sci)

# Fill unmatched with common name match
mask = taxonomy_df["birdnet_idx"].isna()
taxonomy_df.loc[mask, "birdnet_idx"] = taxonomy_df.loc[mask, "common_lower"].map(birdnet_common)

matched   = taxonomy_df[taxonomy_df["birdnet_idx"].notna()]
unmatched = taxonomy_df[taxonomy_df["birdnet_idx"].isna()]

print(f"Matched:   {len(matched)} / {len(taxonomy_df)}")
print(f"Unmatched: {len(unmatched)}")
print(f"\nSample matched:")
print(matched[["scientific_name", "common_name", "birdnet_idx"]].head(10))
print(f"\nSample unmatched:")
print(unmatched[["scientific_name", "common_name"]].head(10))

In [ ]:
# ── How many of our 234 targets are Aves? ─────────────────────
class_breakdown = taxonomy_df[
    taxonomy_df["primary_label"].astype(str).isin(sub_cols)
]["class_name"].value_counts()

print("Target species by class:")
print(class_breakdown)
print(f"\nTotal: {class_breakdown.sum()}")

# Try matching ONLY Aves
aves_targets = taxonomy_df[
    (taxonomy_df["primary_label"].astype(str).isin(sub_cols)) &
    (taxonomy_df["class_name"] == "Aves")
]
aves_targets["sci_lower"] = aves_targets["scientific_name"].str.strip().str.lower()
aves_targets["birdnet_idx"] = aves_targets["sci_lower"].map(birdnet_sci)

matched_aves = aves_targets[aves_targets["birdnet_idx"].notna()]
print(f"\nAves targets matched in BirdNET: {len(matched_aves)} / {len(aves_targets)}")
print(matched_aves[["scientific_name", "primary_label", "birdnet_idx"]].head(10))

In [21]:
# ── Search for Perch in current inputs ───────────────────────
import os
for root, dirs, files in os.walk("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8"):
    for f in files:
        if "perch" in f.lower() or "perch" in root.lower():
            print(os.path.join(root, f))


In [22]:
import os

PERCH_PATH = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8"

print("All Perch files:")
for root, dirs, files in os.walk(PERCH_PATH):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {size:.1f} MB  {path}")

All Perch files:
  3.7 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/saved_model.pb
  0.0 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/tflite_ckpt.txt
  0.0 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/genus.csv
  0.0 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/family.csv
  0.0 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/order.csv
  0.1 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/label.csv
  0.0 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/variables/variables.index
  91.4 MB  /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/

In [23]:
import tensorflow as tf
import numpy as np
import pandas as pd
import librosa

PERCH_PATH = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8"

# Load Perch
perch_model = tf.saved_model.load(PERCH_PATH)
print("✓ Perch loaded")

# Check signatures
print("Signatures:", list(perch_model.signatures.keys()))

# Check labels
label_df = pd.read_csv(f"{PERCH_PATH}/assets/label.csv")
print(f"\nPerch labels shape: {label_df.shape}")
print(label_df.head())

2026-05-17 22:22:49.868802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779056570.061301      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779056570.113864      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779056570.561589      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779056570.561614      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779056570.561617      57 computation_placer.cc:177] computation placer alr

✓ Perch loaded
Signatures: ['serving_default']

Perch labels shape: (10932, 1)
  ebird2021
0   aakspa1
1   abbbab1
2   abbboo2
3   abbsta2
4   abbwar1


In [24]:
# ── Use signature to call Perch ───────────────────────────────
print("Signatures:", list(perch_model.signatures.keys()))

# Try each signature
for sig_name in list(perch_model.signatures.keys()):
    sig = perch_model.signatures[sig_name]
    print(f"\nSignature '{sig_name}':")
    print("  Inputs:")
    for k, v in sig.structured_input_signature[1].items():
        print(f"    {k}: {v}")
    print("  Outputs:")
    for k, v in sig.structured_outputs.items():
        print(f"    {k}: shape={v.shape}")

Signatures: ['serving_default']

Signature 'serving_default':
  Inputs:
    inputs: TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')
  Outputs:
    order: shape=(None, 41)
    embedding: shape=(None, 1280)
    family: shape=(None, 249)
    frontend: shape=(None, 500, 160)
    genus: shape=(None, 2333)
    label: shape=(None, 10932)


In [25]:
# ── Test inference via signature ──────────────────────────────
sig_name = list(perch_model.signatures.keys())[0]
infer_perch = perch_model.signatures[sig_name]

# Try 5s at 32kHz
test_input = tf.zeros([1, 160000], dtype=tf.float32)
try:
    # Try passing as positional
    out = infer_perch(test_input)
    print("Positional call worked:")
    for k, v in out.items():
        print(f"  {k}: shape={v.shape}")
except Exception as e1:
    print(f"Positional failed: {e1}")
    # Try as keyword — use whatever input key name was printed above
    try:
        out = infer_perch(inputs=test_input)
        print("Keyword 'inputs' worked:")
        for k, v in out.items():
            print(f"  {k}: shape={v.shape}")
    except Exception as e2:
        print(f"Keyword 'inputs' failed: {e2}")
        try:
            out = infer_perch(input=test_input)
            print("Keyword 'input' worked:")
            for k, v in out.items():
                print(f"  {k}: shape={v.shape}")
        except Exception as e3:
            print(f"All attempts failed: {e3}")

I0000 00:00:1779056848.337661     148 service.cc:152] XLA service 0x7fdb90008e60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779056848.337718     148 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779056848.337722     148 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779056849.801086     148 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-17 22:27:35.155414: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-17 22:27:35.292630: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-17 22:27:35.474645: E external/local_xl

Positional call worked:
  order: shape=(1, 41)
  embedding: shape=(1, 1280)
  family: shape=(1, 249)
  frontend: shape=(1, 500, 160)
  genus: shape=(1, 2333)
  label: shape=(1, 10932)


In [26]:
# ── Perch embedding extractor ─────────────────────────────────
def extract_perch_embedding(y, sr=32000):
    """
    Extract 1280-dim Perch embedding from audio.
    Perch expects 5s at 32kHz = 160000 samples.
    """
    # Ensure correct length
    target_len = 160000  # 5s at 32kHz
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="reflect")
    else:
        y = y[:target_len]

    input_tensor = tf.constant(y.reshape(1, -1), dtype=tf.float32)
    out = infer_perch(input_tensor)
    return out["embedding"].numpy()[0]  # (1280,)

# ── Test ──────────────────────────────────────────────────────
test_audio = np.random.randn(160000).astype(np.float32)
emb = extract_perch_embedding(test_audio)
print(f"Embedding shape: {emb.shape}")   # (1280,)
print(f"Embedding range: {emb.min():.3f} to {emb.max():.3f}")
print("✓ Perch embedding extraction working")

Embedding shape: (1280,)
Embedding range: -0.204 to 0.698
✓ Perch embedding extraction working


In [27]:
# ── Perch-based classifier ────────────────────────────────────
import torch
import torch.nn as nn

class PerchClassifier(nn.Module):
    """
    Linear head on top of frozen Perch embeddings.
    Input:  (batch, 1280)
    Output: (batch, 234)
    """
    def __init__(self, n_classes=234, embed_dim=1280):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.head(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = PerchClassifier(n_classes=234).to(device)

n_params = sum(p.numel() for p in classifier.parameters())
print(f"Classifier parameters: {n_params:,}")

# Sanity check
dummy = torch.randn(4, 1280).to(device)
out   = classifier(dummy)
print(f"Output shape: {out.shape}")  # (4, 234)
print("✓ PerchClassifier ready")

Classifier parameters: 849,898
Output shape: torch.Size([4, 234])
✓ PerchClassifier ready


In [28]:
# ── Perch-based classifier ────────────────────────────────────
import torch
import torch.nn as nn

class PerchClassifier(nn.Module):
    """
    Linear head on top of frozen Perch embeddings.
    Input:  (batch, 1280)
    Output: (batch, 234)
    """
    def __init__(self, n_classes=234, embed_dim=1280):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.head(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = PerchClassifier(n_classes=234).to(device)

n_params = sum(p.numel() for p in classifier.parameters())
print(f"Classifier parameters: {n_params:,}")

# Sanity check
dummy = torch.randn(4, 1280).to(device)
out   = classifier(dummy)
print(f"Output shape: {out.shape}")  # (4, 234)
print("✓ PerchClassifier ready")

Classifier parameters: 849,898
Output shape: torch.Size([4, 234])
✓ PerchClassifier ready


In [29]:
# ── Perch Dataset ─────────────────────────────────────────────
from torch.utils.data import Dataset

class PerchDataset(Dataset):
    def __init__(self, df, sub_cols, training=True):
        self.df           = df.reset_index(drop=True)
        self.sub_cols     = sub_cols
        self.training     = training
        self.label_to_idx = {col: i for i, col in enumerate(sub_cols)}
        self.n_classes    = len(sub_cols)
        print(f"PerchDataset: {len(self.df)} recordings | training={training}")

    def __len__(self):
        return len(self.df)

    def _build_label_vector(self, primary, secondary=None):
        vec = np.zeros(self.n_classes, dtype=np.float32)
        p = str(primary).strip()
        if p in self.label_to_idx:
            vec[self.label_to_idx[p]] = 1.0
        if secondary is not None:
            try:
                for s in (eval(str(secondary)) 
                          if isinstance(secondary, str) else []):
                    s = str(s).strip()
                    if s in self.label_to_idx:
                        vec[self.label_to_idx[s]] = max(
                            vec[self.label_to_idx[s]], 0.3
                        )
            except Exception:
                pass
        return vec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = CFG.train_audio_dir / str(row["filename"])

        try:
            total_dur = librosa.get_duration(path=str(audio_path))
            if total_dur <= CFG.duration:
                offset = 0.0
            elif self.training:
                offset = np.random.uniform(0, total_dur - CFG.duration)
            else:
                offset = (total_dur - CFG.duration) / 2

            y, sr = librosa.load(str(audio_path), sr=32000, mono=True,
                                 offset=offset, duration=5.0)
        except Exception:
            y = np.zeros(160000, dtype=np.float32)

        # Light augmentation
        if self.training:
            if np.random.random() < 0.3:
                y = y + np.random.uniform(0.0005, 0.002) * np.random.randn(len(y))
                y = y.astype(np.float32)
            if np.random.random() < 0.3:
                gain = 10 ** (np.random.uniform(-6, 6) / 20)
                y = (y * gain).clip(-1.0, 1.0)

        # Extract Perch embedding
        emb = extract_perch_embedding(y)
        label_vec = self._build_label_vector(
            row["primary_label"], row.get("secondary_labels")
        )
        return torch.tensor(emb), torch.tensor(label_vec)

In [ ]:
# ── Train Perch classifier ────────────────────────────────────
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

BATCH_SIZE = 64    # embeddings are small — large batches fine
NUM_EPOCHS = 10
LR         = 1e-3
VAL_SPLIT  = 0.1

criterion = SmoothedBCELoss(smoothing=0.1)

full_ds = PerchDataset(inat_df, list(sub_cols), training=True)
n_val   = int(len(full_ds) * VAL_SPLIT)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))
val_ds.dataset.training = False

# num_workers=0 — TF and PyTorch multiprocessing conflict
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

optimizer = AdamW(classifier.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_auc, best_epoch, history = 0.0, 0, []

for epoch in range(1, NUM_EPOCHS + 1):
    classifier.train()
    train_loss = 0.0
    t0 = time.time()

    for batch_idx, (embs, labels) in enumerate(train_loader):
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(classifier(embs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch} | batch {batch_idx+1}/{len(train_loader)} "
                  f"| loss {loss.item():.4f}")

    train_loss /= len(train_loader)
    classifier.eval()
    val_loss, all_labels, all_probs = 0.0, [], []

    with torch.no_grad():
        for embs, labels in val_loader:
            embs, labels = embs.to(device), labels.to(device)
            logits = classifier(embs)
            val_loss += criterion(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_auc   = compute_auc(np.vstack(all_labels), np.vstack(all_probs))
    scheduler.step()

    print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS} | train={train_loss:.4f} | "
          f"val={val_loss:.4f} | auc={val_auc:.4f} | "
          f"lr={scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.0f}s")

    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_auc": val_auc})

    if val_auc > best_auc:
        best_auc, best_epoch = val_auc, epoch
        torch.save(classifier.state_dict(),
                   "/kaggle/working/perch_classifier.pth")
        print(f"  ✓ Saved (auc={best_auc:.4f})")

    torch.save({"epoch": epoch, "state_dict": classifier.state_dict(),
                "best_auc": best_auc, "history": history},
               "/kaggle/working/perch_checkpoint.pth")

print(f"\nTraining complete. Best AUC={best_auc:.4f} at epoch {best_epoch}")

In [1]:
# Verify best weights are saved
import os
path = "/kaggle/working/perch_classifier.pth"
print(f"Exists: {os.path.exists(path)}")
print(f"Size: {os.path.getsize(path)/1024:.1f} KB")

Exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/perch_classifier.pth'